# Disease Label Generation

**Project:** OneHealth-ML  
**Purpose:** Apply clinical guideline‑based rules to create binary labels for all 7 target diseases.  
**Output:** `data/processed/labeled_master.csv`

## Imports

In [1]:
import pandas as pd
import numpy as np
import os
import sys

sys.path.append("..")

## Paths

In [2]:
INTERIM_DIR = "../data/interim"
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

## Load all verified CSVs

In [3]:
datasets = {}
for fname in os.listdir(INTERIM_DIR):
    if fname.endswith("_verified.csv"):
        key = fname.replace("_verified.csv", "")
        datasets[key] = pd.read_csv(os.path.join(INTERIM_DIR, fname))
        print(f"Loaded {key}: {datasets[key].shape}")

Loaded ALB_CR_2013: (8291, 7)
Loaded ALB_CR_2015: (8608, 9)
Loaded ALB_CR: (7936, 8)
Loaded ALQ_2013: (5924, 11)
Loaded ALQ_2015: (5735, 11)
Loaded ALQ: (5533, 10)
Loaded BIOPRO_2013: (6979, 39)
Loaded BIOPRO_2015: (6744, 39)
Loaded BIOPRO: (6401, 41)
Loaded BMX_2013: (9813, 27)
Loaded BMX_2015: (9544, 27)
Loaded BMX: (8704, 21)
Loaded BPQ_2013: (6464, 15)
Loaded BPQ_2015: (6327, 12)
Loaded BPQ: (6161, 11)
Loaded BPX_2013: (9813, 24)
Loaded BPX_2015: (9544, 22)
Loaded BPX: (8704, 21)
Loaded CBC_2013: (9422, 22)
Loaded CBC_2015: (9165, 22)
Loaded CBC: (8366, 22)
Loaded DBQ_2013: (10175, 52)
Loaded DBQ_2015: (9971, 52)
Loaded DBQ: (9254, 46)
Loaded DEMO_2013: (10175, 48)
Loaded DEMO_2015: (9971, 48)
Loaded DEMO: (9254, 46)
Loaded DIQ_2013: (9770, 55)
Loaded DIQ_2015: (9575, 55)
Loaded DIQ: (8897, 54)
Loaded GHB_2013: (6979, 3)
Loaded GHB_2015: (6744, 3)
Loaded GHB: (6401, 2)
Loaded GLU_2013: (3329, 7)
Loaded GLU_2015: (3191, 5)
Loaded GLU: (3036, 4)
Loaded HDL_2013: (8291, 4)
Loaded HDL_

## Merge into a single master DataFrame
(Inner join on SEQN – keeps only participants present in all 22 datasets)

In [4]:
# Start with DEMO as base
df = datasets["DEMO"].copy()

# List of remaining datasets in a sensible order (questionnaires then labs)
merge_order = [
    "BMX", "GHB", "GLU", "BPX", "BIOPRO",
    "DIQ", "KIQ_U", "SMQ", "ALQ", "PAQ", "DBQ",
    "MCQ", "BPQ", "CBC", "HDL", "TCHOL", "TRIGLY",
    "INS", "HSCRP", "ALB_CR", "HSQ"
]

for ds_name in merge_order:
    df = df.merge(datasets[ds_name], on="SEQN", how="inner")
    print(f"Merged {ds_name}: {df.shape}")

print(f"\nFinal master shape: {df.shape}")
print("Duplicate SEQN:", df["SEQN"].duplicated().sum())

Merged BMX: (8704, 66)
Merged GHB: (6401, 67)
Merged GLU: (3036, 70)
Merged BPX: (3036, 90)
Merged BIOPRO: (3036, 130)
Merged DIQ: (3036, 183)
Merged KIQ_U: (2541, 198)
Merged SMQ: (2541, 234)
Merged ALQ: (2541, 243)
Merged PAQ: (2541, 259)
Merged DBQ: (2541, 304)
Merged MCQ: (2541, 379)
Merged BPQ: (2541, 389)
Merged CBC: (2541, 410)
Merged HDL: (2541, 412)
Merged TCHOL: (2541, 414)
Merged TRIGLY: (2541, 423)
Merged INS: (2541, 427)
Merged HSCRP: (2541, 429)
Merged ALB_CR: (2541, 436)
Merged HSQ: (2541, 444)

Final master shape: (2541, 444)
Duplicate SEQN: 0


## Derived Variables
Compute eGFR (CKD‑EPI 2021) and mean blood pressure.

In [5]:
# eGFR using CKD-EPI 2021 equation (without race coefficient)
def egfr_ckdepi_2021(serum_creatinine, age, sex):
    """
    serum_creatinine in mg/dL, age in years, sex: 1=Male, 2=Female.
    Returns eGFR in mL/min/1.73 m².
    """
    scr = serum_creatinine
    # kappa and alpha parameters
    kappa = np.where(sex == 1, 0.9, 0.7)
    alpha = np.where(sex == 1, -0.302, -0.241)
    # min/max ratio
    ratio = scr / kappa
    ratio_min = np.minimum(ratio, 1)
    ratio_max = np.maximum(ratio, 1)
    
    egfr = 142 * (ratio_min ** alpha) * (ratio_max ** -1.200) \
           * (0.9938 ** age) \
           * np.where(sex == 1, 1.0, 1.012)
    return egfr

# Apply eGFR
df["eGFR"] = egfr_ckdepi_2021(
    serum_creatinine=df["LBXSCR"],
    age=df["RIDAGEYR"],
    sex=df["RIAGENDR"]
)

# Mean blood pressure from available readings
bp_sys_cols = ["BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4"]
bp_dia_cols = ["BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]

df["mean_sbp"] = df[bp_sys_cols].mean(axis=1, skipna=True)
df["mean_dbp"] = df[bp_dia_cols].mean(axis=1, skipna=True)

C:\Users\MiR\AppData\Local\Temp\ipykernel_14052\4014458245.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["eGFR"] = egfr_ckdepi_2021(
C:\Users\MiR\AppData\Local\Temp\ipykernel_14052\4014458245.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["mean_sbp"] = df[bp_sys_cols].mean(axis=1, skipna=True)
C:\Users\MiR\AppData\Local\Temp\ipykernel_14052\4014458245.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Conside

## Disease Label Functions
Define helper functions for each disease according to clinical guidelines.

In [6]:
def label_diabetes(row):
    """ADA 2024: HbA1c ≥ 6.5 OR FPG ≥ 126 OR doctor diagnosed (DIQ010=1). Exclude pregnant."""
    if row["RIDEXPRG"] == 1:
        return np.nan
    if pd.isna(row["LBXGH"]) and pd.isna(row["LBXGLU"]) and pd.isna(row["DIQ010"]):
        return np.nan
    if (row["LBXGH"] >= 6.5) or (row["LBXGLU"] >= 126) or (row["DIQ010"] == 1):
        return 1
    return 0

def label_hypertension(row):
    """ACC/AHA 2017: mean SBP ≥ 130 or mean DBP ≥ 80 or doctor diagnosed (BPQ020=1)."""
    if pd.isna(row["mean_sbp"]) and pd.isna(row["mean_dbp"]) and pd.isna(row["BPQ020"]):
        return np.nan
    if (row["mean_sbp"] >= 130) or (row["mean_dbp"] >= 80) or (row["BPQ020"] == 1):
        return 1
    return 0

def label_ckd(row):
    """KDIGO 2024: eGFR < 60 OR dialysis (KIQ025=1) OR weak kidneys (KIQ022=1)."""
    if pd.isna(row["eGFR"]) and pd.isna(row["KIQ025"]) and pd.isna(row["KIQ022"]):
        return np.nan
    if (row["eGFR"] < 60) or (row["KIQ025"] == 1) or (row["KIQ022"] == 1):
        return 1
    return 0

def label_dyslipidemia(row):
    """NCEP ATP III: TC ≥ 200 OR HDL < 40(M)/50(F) OR TG ≥ 150."""
    if pd.isna(row["LBXTC"]) and pd.isna(row["LBDHDD"]) and pd.isna(row["LBXTR"]):
        return np.nan
    # HDL criteria depend on sex
    hdl_low = (row["LBDHDD"] < 40) if row["RIAGENDR"] == 1 else (row["LBDHDD"] < 50)
    if (row["LBXTC"] >= 200) or hdl_low or (row["LBXTR"] >= 150):
        return 1
    return 0

def label_cvd(row):
    """Any self‑reported history: CHF, CHD, angina, MI, stroke."""
    cols = ["MCQ160B", "MCQ160C", "MCQ160D", "MCQ160E", "MCQ160F"]
    if row[cols].isna().all():
        return np.nan
    if (row[cols] == 1).any():
        return 1
    return 0

def label_metabolic_syndrome(row):
    """IDF/NCEP ATP III: ≥3 of 5 components."""
    components = 0
    # 1. Waist circumference
    if (row["RIAGENDR"] == 1 and row["BMXWAIST"] > 102) or (row["RIAGENDR"] == 2 and row["BMXWAIST"] > 88):
        components += 1
    # 2. Triglycerides ≥ 150
    if row["LBXTR"] >= 150:
        components += 1
    # 3. HDL cholesterol
    hdl_low = (row["LBDHDD"] < 40) if row["RIAGENDR"] == 1 else (row["LBDHDD"] < 50)
    if hdl_low:
        components += 1
    # 4. Blood pressure (SBP ≥ 130 or DBP ≥ 85)
    if not pd.isna(row["mean_sbp"]) and not pd.isna(row["mean_dbp"]):
        if row["mean_sbp"] >= 130 or row["mean_dbp"] >= 85:
            components += 1
    # 5. Fasting glucose ≥ 100
    if row["LBXGLU"] >= 100:
        components += 1
    # If any component variable completely missing, label is NaN
    if pd.isna(row["BMXWAIST"]) or pd.isna(row["LBXTR"]) or pd.isna(row["LBDHDD"]) \
       or pd.isna(row["mean_sbp"]) or pd.isna(row["LBXGLU"]):
        return np.nan
    return 1 if components >= 3 else 0

def label_liver_disease(row):
    """AASLD 2018 early‑risk proxy: elevated liver enzymes + BMI ≥ 25."""
    if pd.isna(row["LBXSATSI"]) and pd.isna(row["LBXSASSI"]) and pd.isna(row["LBXSGTSI"]) and pd.isna(row["BMXBMI"]):
        return np.nan
    # Upper limits
    alt_uln = 40 if row["RIAGENDR"] == 1 else 30
    ast_uln = 40  # typically 40 for both sexes
    ggt_uln = 50
    enzyme_elevated = (row["LBXSATSI"] > alt_uln) | (row["LBXSASSI"] > ast_uln) | (row["LBXSGTSI"] > ggt_uln)
    bmi_high = row["BMXBMI"] >= 25
    if enzyme_elevated and bmi_high:
        return 1
    return 0

## Apply Label Functions
Create a binary label column for each disease.

In [7]:
df["label_diabetes"] = df.apply(label_diabetes, axis=1)
df["label_hypertension"] = df.apply(label_hypertension, axis=1)
df["label_ckd"] = df.apply(label_ckd, axis=1)
df["label_dyslipidemia"] = df.apply(label_dyslipidemia, axis=1)
df["label_cvd"] = df.apply(label_cvd, axis=1)
df["label_metabolic_syndrome"] = df.apply(label_metabolic_syndrome, axis=1)
df["label_liver_disease"] = df.apply(label_liver_disease, axis=1)

C:\Users\MiR\AppData\Local\Temp\ipykernel_14052\1474595990.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["label_diabetes"] = df.apply(label_diabetes, axis=1)
C:\Users\MiR\AppData\Local\Temp\ipykernel_14052\1474595990.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["label_hypertension"] = df.apply(label_hypertension, axis=1)
C:\Users\MiR\AppData\Local\Temp\ipykernel_14052\1474595990.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has

## Label Summary

In [8]:
label_cols = [c for c in df.columns if c.startswith("label_")]
print("Label counts (1 = positive, 0 = negative, NaN = excluded):")
for col in label_cols:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))

Label counts (1 = positive, 0 = negative, NaN = excluded):

label_diabetes:
label_diabetes
0.0    1937
1.0     583
NaN      21
Name: count, dtype: int64

label_hypertension:
label_hypertension
1    1475
0    1066
Name: count, dtype: int64

label_ckd:
label_ckd
0    2294
1     247
Name: count, dtype: int64

label_dyslipidemia:
label_dyslipidemia
1.0    1388
0.0    1011
NaN     142
Name: count, dtype: int64

label_cvd:
label_cvd
0    2221
1     320
Name: count, dtype: int64

label_metabolic_syndrome:
label_metabolic_syndrome
0.0    1342
1.0     850
NaN     349
Name: count, dtype: int64

label_liver_disease:
label_liver_disease
0.0    2174
1.0     361
NaN       6
Name: count, dtype: int64


## Save Labeled Master Dataset

In [9]:
output_path = os.path.join(PROCESSED_DIR, "labeled_master.csv")
df.to_csv(output_path, index=False)
print(f"Labeled master dataset saved to {output_path}")
print(f"Shape: {df.shape}")

Labeled master dataset saved to ../data/processed\labeled_master.csv
Shape: (2541, 454)
